# 2HRX9P6HKXA8V

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
import tabulate
from IPython.display import display, Markdown

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

Preemptively set new Pandas option, also set matplotlib to close

In [ ]:
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

Allow reloading of custom Python classes without resetting kernel

In [ ]:
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged
%store -r restaurants_by_4m_coverage
%store -r time_differences
%store -r time_differences_details
%store -r before_after_details_true

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

Time Differences

In [ ]:
%store -r restaurant_data_unprocessed
timezones_acronyms = {}
for loc_id, df in restaurant_data_unprocessed.items():
    time = df['created_at'].iloc[0]
    timezone = time.strip('0123456789-+: ')
    timezones_acronyms[loc_id] = timezone
timezones = {
    '0RJH3FFPYBPEY': 'America/New_York',
    '1SQPTEGYPH0GA': 'America/Denver',
    '3AXDVZJYN9DRS': 'Europe/London',
    '75WYSXR9QBK5M': 'Pacific/Honolulu',
    '78AY09MVJVTYE': 'America/New_York',
    '9XKJD8DQTH559': 'America/New_York',
    'AQD04SM0J92WA': 'America/Los_Angeles',
    'CB2KHY1C2G9PT': 'America/New_York',
    'EMBVNVD207CC6': 'America/New_York',
    'JHDN7CF1C03X5': 'America/Chicago',
    'L3XS7WSJ4AJA3': 'Europe/London',
    'L69HYJ4Y3TR91': 'America/New_York',
    'LBMCPAYT7W36V': 'America/New_York',
    'LBZEEFSBJNB3Z': 'America/Los_Angeles',
    'LFZFT3VASXPED': 'Australia/Sydney',
    'LQ5EH4BKGV61T': 'America/New_York',
    'LZ5MR1TS37E7W': 'America/Los_Angeles',
    'MS8R16DY0JQAM': 'America/Los_Angeles',
    'N0PC58FB2XAZ3': 'America/Chicago',
    'S8MT0YGD2KTN9': 'America/New_York',
    'SAFK7ND1HR6XS': 'America/Los_Angeles',
    'SRQS8F7JWA9MZ': 'America/New_York',
    'V3Q26BHF3SE2H': 'America/New_York',
    'W8T41JZK0ZMEP': 'America/New_York',
    'WJA3YCD4QBWRX': 'America/New_York',
    '1G5AJ17XCH2A8': 'America/Chicago',
    'ADPFRN3QZRCXK': 'America/Los_Angeles',
    'ED5J990H5VAZT': 'America/Los_Angeles',
    '2HRX9P6HKXA8V': 'America/Los_Angeles',
    'C0BE4NDSW26QN': 'America/New_York'
}
for loc_id, df in sales_and_menu_data.items():
    df.index = df.index.tz_convert(timezones[loc_id])

before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'] = before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'].str.title()

In [ ]:
loc_id = '2HRX9P6HKXA8V'
df = sales_and_menu_data[loc_id]
df = df.assign(item_modifications = lambda df: df['item_modifications'].str.title())

In [ ]:
time_differences_details[loc_id]

10 Hour Difference

In [ ]:
df['item_name'].value_counts().sort_values(ascending=False).head(10)

In [ ]:
df.loc[pd.Timestamp('2019-07-17 4:45:58+00:00'):pd.Timestamp('2019-07-18').tz_localize('UTC')].head(5)

20 Hour Difference

In [ ]:
df.loc[pd.Timestamp('2020-06-07 22:27:20+00:00'):pd.Timestamp('2020-06-09').tz_localize('UTC')].head(10)

In [ ]:
time_differences['2HRX9P6HKXA8V']

In [ ]:
time_differences_details['2HRX9P6HKXA8V'][0][time_differences_details['2HRX9P6HKXA8V'][0] == 5]

In [ ]:
name_changes = {
    #"Veggie Wurst": ["Vegetarian"],
    "Potato Chips": ["Tim's Cascade Potato Chips", "Tim'S Cascade Potato Chips", "Kettle Brand Potato Chips", "Potato Chips", "Chips"], # 
    "Big Bob Bratwurst": ["Big Bob"],
    "Warm Bavarian Pretzel": ["Bavarian Pretzel", "Pretzel"],
    "Hans Jalapeno & Cheddar": ["Han's Jalapeno & Cheddar", "Jalapeno & Cheddar", "Jalapeño & Cheddar", "Hans' Jalape√±O & Cheddar", "Jalape√±O & Cheddar"],
    "Dirtyface Beer Wurst": ["Beer Wurst"],
    "Spinach Organic Chicken": [], # "Chicken", "Organic Chicken"
    "Italian Organic Chicken" : [],
    "Organic Chicken Sausage" : [],
    "Helgas Giant Kelbassi": ["Helga's Giant Kelbassi", "Kelbassi", "Giant Kelbassi", "Helga'S Giant Kelbassi"],
    "Large Sauerkraut - 8Oz Bowl": ["Side Sauerkraut - 8Oz Bowl", "Side Saurkraut"],
    "Gluhwein": ["Glühwein"],
    "Turkey Dog": ["Organic Turkey Dog"],
    "Gingerbread Cookie": ["Haus Made Gingerbread Cookie"],
    "Big City Beef Frank" : ["Big City"],
    "Vegan Soup": ["House Vegan Soup"], # "Veggie Soup" 
    #"Egift Card": ["Gift Card", "Promotional $5 Gift Certificates", "Donation $5 Gift Certificates"],
    "Bottled Water": ["Athena Bottled Water"],
    "Pepsi Bottled Sodas": ["Pepsi Fountain", "Diet Pepsi Fountain", "Pepsi", "Diet Pepsi", "Pepsi Fountain Sodas", "Pepsi Bottled Sodas 20Oz"],
    "Dr. Pepper Fountain": ["Dr. Pepper"],
    "7-Up Fountain": ["7-Up", "-Up Fountain", "-Up"],
    "Mountain Dew Fountain": ["Mountain Dew"],
    "Rootbeer Fountain": ["Rootbeer"],
}

# Swap the keys and values
name_changes_dict = {variant: canonical for canonical, variants in name_changes.items() for variant in variants}

# Item names to swap based on modications
modification_name_changes = [('Beyond Sausage', 'Carne', 'Beyond Sausage With Chile Con Carne'),
                             ('Beyond Sausage', 'Cream|Cheese|Mayo|Beech', 'Beyond Sausage With Dairy'),
                             ('Veggie Wurst', 'Vegan', 'Vegan Veggie Wurst'),
                             ('Veggie Wurst', 'Carne', 'Veggie Wurst With Chile Con Carne'),
                             ('Vegetarian', 'Vegan', 'Vegan Vegetarian'),
                             ('Vegetarian', 'Carne', 'Vegetarian With Chile Con Carne'),
                             ('Vegan Chili', 'Cream|Cheese|Beech', 'Vegetarian Chili'),
                             ('Veggie Soup', 'Cream|Cheese|Beech', 'Vegetarian Soup'),
                             ('Potato Chips', 'Cheddar', 'Cheddar Potato Chips')]

# Turn into dataframe for viewing
modification_name_changes_df = pd.DataFrame(data = modification_name_changes, columns = ['name', 'modification', 'new_name'])

alcohol_changes = {
    "Icicle Premium Pilsner": [],
    "Dirtyface Amber Lager": ["Dirtyface Beer Wurst", "Dirtyface Amber Mustard", "To-Go Dirtyface Amber Single 16Oz Can", "To-Go Dirtyface Amber 22Oz Bottle", "To-Go Dirtyface Amber 4 Pack 16Oz Cans", "Bottled Dirtyface"],
    "Bootjack IPA": ["Bootjack Ipa", "To-Go Bootjack Ipa Single Can 12Oz", "To-Go Bootjack Ipa 6 Pack 12Oz"],
    "Alpenhaze Hazy IPA": ["Alpenhaze"],
    "Colchuck Raspberry Wheat": ["To-Go Colchuck Raspberry Wheat Single Can 16Oz", "To-Go Colchuck Raspberry Wheat 4 Pack 16Oz", "Raspberry Dark Persuasion"],
    "Dark Persuasion": ["Dark Persuasion Chocolate Cake Ale", "To-Go Dark Persuasion German Chocolate Cake Ale Single Can 12Oz", "To-Go Dark Persuasion German Chocolate Cake Ale 6 Pack 12Oz"],
    "Hofbräu Original": ["Hofbr√§U Original"],
    "Hofbräu Hefe Weizen": ["Hofbrau Hefeweizen", "Hofbr√§U Hefeweizen", "Drubru Hefeweizen"],
    "Hofbräu Dunkel": ["Hofbr√§U Dunkel", "Hofbr√§U Dunkle", "Hofbrau Dunkel"],
    "Yonder Vantage Semi-Sweet Cider": ["Trailbreaker Cider 12Oz Can", "To-Go Trailbreaker Cider 12Oz Can", "Trailbreaker Cider 12Oz Can - Dine-In", "To-Go Trailbreaker Cider 12Oz Can *Takeout Only*", "Pitcher Draft Cider"],
    "Quartet Bordeaux-Style Blend": ["Cellars Trio", "Cellars Quartet", "Cellars Trio Bottle"],
    "Montage": ["Eagle Creek Montage (Merlot)", "Eagle Creek Montage", "Eagle Creek Montage Bottle"],
    "Chardonnay": ["Milbrandt Chardonnay Bottle"],
    "Pinot Grigio": ["Eagle Creek Pinot Grigio Bottle"],
    "Riesling": ["Ryan Patrick Riesling Bottle"],
    "Gewürztraminer": ["Icicle Ridge Gewurztraminer Bottle"],
    "Rosé of Sangiovese": ["Kestrel Ros√© Bottle", "Maryhill Ros√©", "Maryhill Ros√© Bottle"],
    "Ghostfish Brewing Company": ["Ghostfish Gf Can 12Oz"],
    "Athletic Brewing IPA": ["Athletic Brewing Ipa *Non-Alcoholic* 12Oz Can", "To-Go Athletic Brewing Ipa *Non-Alcoholic* 12Oz Can"],
    "Athletic Golden Ale": ["Athletic Brewing Blonde Ale *Non-Alcoholic & Gluten Free* 12Oz Can", "To-Go Athletic Brewing Blonde Ale *Non-Alcoholic & Gluten Free* 12Oz Can"],
    "Bitburger Drive Pilsner": ["N/A Beer - Bitburger"],
    "Crosscut Pilsner": ["To-Go Crosscut Pilsner Single Can 16Oz", "To-Go Crosscut Pilsner 4 Pack 16Oz"],
    "Kickstand Citra Pale Ale": ["Kickstand Pale Ale", "To-Go Kickstand Pale Ale Single Can 12Oz", "To-Go Kickstand Pale Ale 6 Pack 12Oz"],
    "Timbertown Brown": [],
    "Snow Creek K√∂Lsch": [],
    "Leavenworth Festbier": [],
    "Pamm'S American Lager": [],
    "Knock Off Australian Lager": [],
    "Enchantments Hazy Ipa": ["To-Go Enchantments Hazy Ipa Single Can 16Oz", "To-Go Enchantments Hazy Ipa Single Can 12Oz", "To-Go Enchantments Hazy Ipa 4 Pack 16Oz", "To-Go Enchantments Hazy Ipa 6 Pack 12Oz"],
    "One In Eight Fresh Hop Ipa": [],
    "Drumfire Dark Lager": [],
    "Drubru Kolsch": ["Drubru K√∂Lsch"],
    "Icicle Lager": [],
    "Gluten Free Beer": ["Gluten Free 16Oz - Dine-In", "To-Go Gluten Free 16Oz"],
    "N/A Beer": [],
    "Drubru Hefeweizen": ["Dru Bru K√∂Lsch"],
    "Sawdog IPA": ["Sawdog"],
    "Ryan Patrick Riesling Bottle": ["Riesling"],
}

non_alcoholic_drinks = [
    "Lemonade",
    "Pepsi Bottled Sodas",
    "Bottled Water",
    "Iced Tea",
    "Hot Cocoa",
    "Dr. Pepper Fountain",
    "Hot Tea",
    "Rootbeer Fountain",
    "7-Up Fountain",
    "Apple Juice",
    "Coffee",
    "Mountain Dew Fountain",
    "Pepsi Fountain Sodas 22Oz",
    "Gatorade Fountain",
    "Bottled Soda",
    "Gatorade",
    "Common Ground Coffee Amber",
    "Tap Water To-Go",
    "Fountain Refill"
]

merch = ["Souvenir Water Bottle",
         "Souvenir Pint Glass",
         "Souvenir Wine Glass",
         "Gift Card",
    "Royal Blue Tee",
    "Black Tee",
    "Trucker Hat",
    "Dog Cookie (Dog Treat)",
    "Egift Card",
    "Black Beanie Winter Hat",
    "Blue Zip Up Sweatshirt",
    "T-Shirt Blue *Sale* Limited Sizes",
    "Donation $5 Gift Certificates",
    "Keychain Bottle Opener",
    "Reusable Straw",
    "Re-Useable Straw",
    "Promotional $5 Gift Certificates",
    "Ben Davis Button Up",
    "Shipping Charge",
    "Cowbell",
    "Corkage Fee",
    'Magnet', 
    'Carryout Paper Bag', 
    'Reusable Tote Bag', 
    'Sticker',
    'Winter Pom Pom Hat', 
    'Gray Pullover Sweatshirt',
    'V-Neck Tee', 
    'Face Buff', 
    'Winter Hat',
    'Sleeve Baseball Tee - Black On Black',
    'Black/Gray Zip Up Sweatshirt',
    'Gray Pull Sweatshirt',
    'Blue Waffle Beanie Winter Hat',
    'Keychain',
    'T-Shirt Blue',
    'Scarf',
    'Blue Zip Sweatshirt',
    'Sleeve Baseball Tee - Black On Gray', 
    'Dog Treat',
    'Foodles', 
    'Black Winter Hat', 
    'Mountain Equipment Jacket',
    'Patch Logo', 
    'To-Go Refill', 
    'Black Beanie',
    'Christmas Sweater 20',
    'Mountain Equipment Vest'
]

rare =[]

unknown = []

vegetarian = ['Warm Bavarian Pretzel', 
              'Beyond Sausage With Dairy'
              'Vegetarian', 
              'Veggie Wurst', 
              'Vegetarian Chili', 
              'Gingerbread Cookie',
              'Carrots & Ranch',
              'Cheddar Potato Chips'
              'Vegetarian Soup']

vegan = ['Vegan Vegetarian', 
         'Vegan Veggie Wurst', 
         'Beyond Sausage',
         'Veggie Soup', 
         'Vegan Soup',
         'Vegan Chili',
         'Apple Slices',
         'Potato Chips', 
         'Large Sauerkraut - 8Oz Bowl']

alcoholic_drinks = list(alcohol_changes.keys()) + ["Ibc 4 Pack Cans 16Oz", "Ibc 6 Pack Cans", "Ibc 6 Pack Cans 12Oz", "To-Go Single Cans Ibc 16Oz", "To-Go Single Cans Ibc 12Oz"]

others = {}

# Swap the keys and values
replacement_dict = {variant: canonical for canonical, variants in name_changes.items() for variant in variants}
alcohol_replacement_dict = {variant: canonical for canonical, variants in alcohol_changes.items() for variant in variants}

# Items to remove
items_to_remove = []

In [ ]:
df_cleaned = (df
              .assign(item_name=lambda df: df['item_name']
                      .str.strip('123456789./\\ ')  # Clean up item names
                      .replace(replacement_dict)    # Replace names based on dictionary
                      .replace(alcohol_replacement_dict)
                      #.replace(items_to_remove, pd.NA)  # Replace non-dish items with NA
              )
              #.dropna(subset=['item_name'])
              .assign(item_name = lambda df: np.select(condlist = [df['item_name'].eq(name) &  
                                                                   df['item_modifications'].str.contains(modification) for name, modification, _ in modification_name_changes],
                                                       choicelist = modification_name_changes_df['new_name'].tolist(),
                                                       default = df['item_name']),
                      dish_category = lambda df: df['dish_category']
                              .mask(df['item_name'].isin(alcoholic_drinks), 'Alcohol')
                              .mask(df['item_name'].isin(merch), 'Merch')
                              .mask(df['item_name'].isin(non_alcoholic_drinks), 'Drink'),
                      vegetarian = lambda df: df['item_name'].isin(vegetarian + vegan + non_alcoholic_drinks + alcoholic_drinks),
                      vegan = lambda df: df['item_name'].isin(vegan + non_alcoholic_drinks + alcoholic_drinks)
                )
              #.drop('unique_id', axis=1)
             )

df = df_cleaned
food_df = df.query('~dish_category.isin(["Alcohol", "Drink", "Merch"])')

In [ ]:
# Visualizing with gaps for inactive weeks
introduction_fig, ax = plt.subplots(figsize=(14, 8))

# Index into the promotional items for this restaurant
promo_datetime = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert('UTC')

top_n = 30

unique_dishes = (food_df
                 ['item_name']
                 .value_counts()
                 .to_frame(name='c')
                 [:top_n]
                 .index[::-1]
                 )

legend_handles = []

for dish in unique_dishes:
    
    dish_df = food_df.query('item_name == @dish')
    
    vmin = dish_df['unit_price'].min()
    vmax = dish_df['unit_price'].max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = cm.ScalarMappable(norm=norm, cmap='magma')
    
    weekly_quantities = (dish_df
                         .resample('W')
                         .agg({'item_quantity': 'sum', 'unit_price': 'mean'})
                         .query('0 < item_quantity')
                         .assign(week = lambda df: df.index.tz_localize(None).to_period('W'))
                         .set_index('week')
                         )
    
    # For every active week
    for week, row  in weekly_quantities.iterrows():
        
        weekly_quantity = row['item_quantity']
        dot_size = weekly_quantity/70 + 2.5
        weekly_price = row['unit_price']
        color = cmap.to_rgba(weekly_price)

        # Place a blue dot
        ax.hlines(y=dish, xmin=week.start_time, xmax=week.end_time, colors=color, lw=dot_size, label=loc_id)
        
    ax.text(x=food_df.index[-1] + pd.DateOffset(100), y=dish, s=f'${vmin/100:.2f}-${vmax/100:.2f}', verticalalignment='center', horizontalalignment='left', fontsize='x-small', color='gray')
    
    # Create a custom legend entry for this dish
    #color_patch_min = mpatches.Patch(color=cmap.to_rgba(vmin), label=f'{dish} Min: ${vmin/100:.2f}')
    #color_patch_max = mpatches.Patch(color=cmap.to_rgba(vmax), label=f'{dish} Max: ${vmax/100:.2f}')
    #legend_handles.extend([color_patch_min, color_patch_max])

# Place a red circle for the promotional item
ax.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
ax.set_title(f'Weekly Sales of Top {top_n} Dishes for {loc_id}')
ax.set_xlabel('Date')
ax.set_ylabel('Dish')
#ax.legend(handles=legend_handles, title="Price Range per Dish", fontsize='small', loc='upper left', bbox_to_anchor=(1, 1))

# Figure
introduction_fig.tight_layout(rect=[0, 0, 0.85, 1])

plt.show()

In [ ]:
df.query('dish_category == "Drink"')['item_name'].value_counts()

In [ ]:
df